In [11]:
import pandas as pd
import numpy as np
from pprint import pprint
import requests
import json
import bs4
from bs4 import BeautifulSoup
import time
import datetime

# Race Schedule

In [76]:
current_schedule_df = pd.read_csv('raceSchedule.csv', index_col=0)
last_season_collected = current_schedule_df['season'][:-1].iloc[-1]
print(last_season_collected)
currentYear = datetime.datetime.now().year
print(currentYear)

2024
2024


In [72]:
if last_season_collected < currentYear:
    print("Updating the schedule")
    schedule = {'season': [],
            'round': [],
            'raceName': [],
            'circuitId': [],
            'circuitName': [],
            'lat': [],
            'long': [],
            'locality': [],
            'country': [],
            'date': [],
            'sprint': [],
            'url': []}


    for year in list(range(last_season_collected+1,currentYear+1)):
    
        url = 'https://ergast.com/api/f1/{}.json'
        req = requests.get(url.format(year),verify=False)
        races = req.json()
    
        for item in races['MRData']['RaceTable']['Races']:
            try:
                schedule['season'].append(int(item['season']))
            except:
                schedule['season'].append(None)
                
            try:
                schedule['round'].append(int(item['round']))
            except:
                schedule['round'].append(None)
                
            try:
                schedule['raceName'].append(item['raceName'])
            except:
                schedule['raceName'].append(None)
                
            try:
                schedule['circuitId'].append(item['Circuit']['circuitId'])
            except:
                schedule['circuitId'].append(None)
                
            try:
                schedule['circuitName'].append(item['Circuit']['circuitName'])
            except:
                schedule['circuitName'].append(None)
                
            try:
                schedule['lat'].append(float(item['Circuit']['Location']['lat']))
            except:
                schedule['lat'].append(None)
                
            try:
                schedule['long'].append(float(item['Circuit']['Location']['long']))
            except:
                schedule['long'].append(None)
                
            try:
                schedule['locality'].append(item['Circuit']['Location']['locality'])
            except:
                schedule['locality'].append(None)
                
            try:
                schedule['country'].append(item['Circuit']['Location']['country'])
            except:
                schedule['country'].append(None)
                
            try:
                schedule['date'].append(item['date'])
            except:
                schedule['date'].append(None)
                
            try:
                schedule['sprint'].append(item['Sprint']['date'])
            except:
                schedule['sprint'].append(None)
                
            try:
                schedule['url'].append(item['url'])
            except:
                schedule['url'].append(None)

    new_schedule_df = pd.DataFrame(schedule)
    updated_schedule_df = pd.concat([current_schedule_df, new_schedule_df], ignore_index=True)
    updated_schedule_df.to_csv('raceSchedule.csv', index=True)
    print("Schedule Update Finished")
else:
    print("Schedule is up to date")

Schedule is up to date


# Race Results

In [84]:
race_df = pd.read_csv('raceResults.csv', index_col=0)
last_season_collected = race_df['season'][:-1].iloc[-1]
last_round_collected = race_df['round'][:-1].iloc[-1]
print(last_season_collected, last_round_collected)
rounds_df = current_schedule_df[current_schedule_df['season']>=last_season_collected].groupby('season')['round'].apply(list).reset_index(name="rounds")
print(rounds_df)

2022 22
   season                                             rounds
0    2022  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...
1    2023  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...
2    2024  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...


In [89]:
#schedule_df = pd.read_csv('raceSchedule.csv')
#rounds_df = schedule_df.groupby('season')['round'].apply(list).reset_index(name="rounds")
results = {'season': [],
          'round': [],
          'circuitId': [],
          'position': [],
          'number': [],
          'driverId': [],
          'constructorId': [],
          'grid': [],
          'time': [],
          'status': [],
          'points': []}
for index in rounds_df.index:
    for rnd_num in rounds_df['rounds'][index]:
        #print(rounds_df['season'][index],rnd_num)
        if (rounds_df['season'][index] == last_season_collected and rnd_num>last_round_collected) or (rounds_df['season'][index] > last_season_collected):
            url = 'http://ergast.com/api/f1/{}/{}/results.json'
            r = requests.get(url.format(rounds_df['season'][index],rnd_num),timeout=None,verify=False)
            json = r.json()
            try: 
                for item in json['MRData']['RaceTable']['Races'][0]['Results']:
                    try:
                        results['season'].append(int(json['MRData']['RaceTable']['Races'][0]['season']))
                    except:
                        results['season'].append(None)

                    try:
                        results['round'].append(int(json['MRData']['RaceTable']['Races'][0]['round']))
                    except:
                        results['round'].append(None)

                    try:
                        results['circuitId'].append(json['MRData']['RaceTable']['Races'][0]['Circuit']['circuitId'])
                    except:
                        results['circuitId'].append(None)

                    try:
                        results['position'].append(int(item['position']))
                    except:
                        results['position'].append(None)

                    try:
                        results['number'].append(int(item['number']))
                    except:
                        results['number'].append(None)

                    try:
                        results['driverId'].append(item['Driver']['driverId'])
                    except:
                        results['driverId'].append(None)

                    try:
                        results['constructorId'].append(item['Constructor']['constructorId'])
                    except:
                        results['constructorId'].append(None)

                    try:
                        results['grid'].append(int(item['grid']))
                    except:
                        results['grid'].append(None)

                    try:
                        results['time'].append(item['Time']['time'])
                    except:
                        results['time'].append(None)

                    try:
                        results['status'].append(item['status'])
                    except:
                        results['status'].append(None)

                    try:
                        results['points'].append(float(item['points']))
                    except:
                        results['points'].append(None)
            except:
                print("Results not available yet")

Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet


In [90]:
results_df = pd.DataFrame(results)
#results_df.to_csv('raceResults.csv')

In [92]:
results_df.tail()

,season,round,circuitId,position,number,driverId,constructorId,grid,time,status,points
554,2024,6,miami,16,77,bottas,sauber,16,+52.356,Finished,0.0
555,2024,6,miami,17,18,stroll,aston_martin,11,+55.173,Finished,0.0
556,2024,6,miami,18,23,albon,williams,14,+1:16.091,Finished,0.0
557,2024,6,miami,19,20,kevin_magnussen,haas,18,+1:24.683,Finished,0.0
558,2024,6,miami,20,2,sargeant,williams,17,None,Collision,0.0


# Qualifying Results

Part-I: Leveraging F1 official archive to get the qualifying results from the year 1983 to 2002

In [14]:
qualifying_results = pd.DataFrame()
for year in list(range(1983,2003)):
    url = 'https://www.formula1.com/en/results.html/{}/races.html'
    r = requests.get(url.format(year),timeout=None)
    soup = BeautifulSoup(r.text, 'html.parser')
    
    year_links = []
    for page in soup.find_all('a', attrs = {'class':"resultsarchive-filter-item-link FilterTrigger"}):
        link = page.get('href')
        if f'/en/results.html/{year}/races/' in link: 
            year_links.append(link)

    year_df = pd.DataFrame()
    new_url = 'https://www.formula1.com{}'
    for n, link in list(enumerate(year_links)):
        link = link.replace('race-result.html', 'qualifying-0.html')
        df = pd.read_html(new_url.format(link))
        df = df[0]
        df['season'] = year
        df['round'] = n+1
        for col in df:
            if 'Unnamed' in col:
                df.drop(col, axis = 1, inplace = True)

        year_df = pd.concat([year_df, df])

    qualifying_results = pd.concat([qualifying_results, year_df])
    
print(qualifying_results.shape)

(8505, 8)


In [15]:
qualifying_results.tail()

,Pos,No,Driver,Car,Time,season,round,Laps
15,16,12,Olivier Panis PAN,BAR Honda,1:34.192,2002,17,10.0
16,17,17,Pedro de la Rosa DLR,Jaguar Cosworth,1:34.227,2002,17,12.0
17,18,25,Allan McNish MCN,Toyota,1:35.191,2002,17,5.0
18,19,23,Mark Webber WEB,Minardi Asiatech,1:35.958,2002,17,11.0
19,20,22,Alex Yoong YOO,Minardi Asiatech,1:36.267,2002,17,12.0


In [16]:
qualifying_results.to_csv('qualifying_results_I.csv')

Part-II: Utilizing the Ergast Developer API which supports qualifying data from 2003 to current date

In [93]:
#schedule_df = pd.read_csv('raceSchedule.csv')
#rounds_df = schedule_df.groupby('season')['round'].apply(list).reset_index(name="rounds")
quali_results = {'season': [],
          'round': [],
          'circuitId': [],
          'position': [],
          'number': [],
          'driverId': [],
          'constructorId': [],
          'Q1': [],
          'Q2': [],
          'Q3': []}

for index in rounds_df.loc[rounds_df['season'] >= 2003].index:
    for rnd_num in rounds_df['rounds'][index]:
        #print(rounds_df['season'][index],rnd_num)
        if (rounds_df['season'][index] == last_season_collected and rnd_num>last_round_collected) or (rounds_df['season'][index] > last_season_collected):
            url = 'http://ergast.com/api/f1/{}/{}/qualifying.json'
            r = requests.get(url.format(rounds_df['season'][index],rnd_num),timeout=None,verify=False)
            json = r.json()
            try: 
                for item in json['MRData']['RaceTable']['Races'][0]['QualifyingResults']:
                    try:
                        quali_results['season'].append(int(json['MRData']['RaceTable']['Races'][0]['season']))
                    except:
                        quali_results['season'].append(None)

                    try:
                        quali_results['round'].append(int(json['MRData']['RaceTable']['Races'][0]['round']))
                    except:
                        quali_results['round'].append(None)

                    try:
                        quali_results['circuitId'].append(json['MRData']['RaceTable']['Races'][0]['Circuit']['circuitId'])
                    except:
                        quali_results['circuitId'].append(None)

                    try:
                        quali_results['position'].append(int(item['position']))
                    except:
                        quali_results['position'].append(None)

                    try:
                        quali_results['number'].append(int(item['number']))
                    except:
                        quali_results['number'].append(None)

                    try:
                        quali_results['driverId'].append(item['Driver']['driverId'])
                    except:
                        quali_results['driverId'].append(None)

                    try:
                        quali_results['constructorId'].append(item['Constructor']['constructorId'])
                    except:
                        quali_results['constructorId'].append(None)

                    try:
                        quali_results['Q1'].append(item['Q1'])
                    except:
                        quali_results['Q1'].append(None)

                    try:
                        quali_results['Q2'].append(item['Q2'])
                    except:
                        quali_results['Q2'].append(None)

                    try:
                        quali_results['Q3'].append(item['Q3'])
                    except:
                        quali_results['Q3'].append(None)
            except:
                print("Results not available yet")

Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet
Results not available yet


In [94]:
quali_results_df = pd.DataFrame(quali_results)
#quali_results_df.to_csv('qualifying_results_II.csv')

In [97]:
quali_results_df.head()

,season,round,circuitId,position,number,driverId,constructorId,Q1,Q2,Q3
0,2023,1,bahrain,1,1,max_verstappen,red_bull,1:31.295,1:30.503,1:29.708
1,2023,1,bahrain,2,11,perez,red_bull,1:31.479,1:30.746,1:29.846
2,2023,1,bahrain,3,16,leclerc,ferrari,1:31.094,1:30.282,1:30.000
3,2023,1,bahrain,4,55,sainz,ferrari,1:30.993,1:30.515,1:30.154
4,2023,1,bahrain,5,14,alonso,aston_martin,1:31.158,1:30.645,1:30.336


# Driver Standings

In [5]:
driver_standings = {'season': [],
          'round': [],
          'position': [],
          'driverId': [],
          'constructorId': [],
          'points': [],
          'wins': []}

for index in rounds_df.index:
    for rnd_num in rounds_df['rounds'][index]:
        #print(rounds_df['season'][index],rnd_num)
        url = 'http://ergast.com/api/f1/{}/{}/driverStandings.json'
        r = requests.get(url.format(rounds_df['season'][index],rnd_num),timeout=None,verify=False)
        json = r.json()
        try: 
            for item in json['MRData']['StandingsTable']['StandingsLists'][0]['DriverStandings']:
                try:
                    driver_standings['season'].append(int(json['MRData']['StandingsTable']['StandingsLists'][0]['season']))
                except:
                    driver_standings['season'].append(None)

                try:
                    driver_standings['round'].append(int(json['MRData']['StandingsTable']['StandingsLists'][0]['round']))
                except:
                    driver_standings['round'].append(None)

                try:
                    driver_standings['position'].append(int(item['position']))
                except:
                    driver_standings['position'].append(None)

                try:
                    driver_standings['driverId'].append(item['Driver']['driverId'])
                except:
                    driver_standings['driverId'].append(None)

                try:
                    driver_standings['constructorId'].append(item['Constructors'][0]['constructorId'])
                except:
                    driver_standings['constructorId'].append(None)

                try:
                    driver_standings['points'].append(float(item['points']))
                except:
                    driver_standings['points'].append(None)

                try:
                    driver_standings['wins'].append(int(item['wins']))
                except:
                    driver_standings['wins'].append(None)
        except:
            print("Results not available yet")

In [6]:
driver_standings_df = pd.DataFrame(driver_standings)
driver_standings_df.to_csv('driverStandings.csv')

In [7]:
driver_standings_df.tail()

,season,round,position,driverId,constructorId,points,wins
28037,2023,22,18,zhou,alfa,6.0,0
28038,2023,22,19,kevin_magnussen,haas,3.0,0
28039,2023,22,20,lawson,alphatauri,2.0,0
28040,2023,22,21,sargeant,williams,1.0,0
28041,2023,22,22,de_vries,alphatauri,0.0,0


# Constructors Standings

In [3]:
schedule_df = pd.read_csv('raceSchedule.csv')
rounds_df = schedule_df.groupby('season')['round'].apply(list).reset_index(name="rounds")

In [7]:
constructor_standings = {'season': [],
          'round': [],
          'position': [],
          'constructorId': [],
          'points': [],
          'wins': []}

for index in rounds_df.index:
    for rnd_num in rounds_df['rounds'][index]:
        url = 'http://ergast.com/api/f1/{}/{}/constructorStandings.json'
        r = requests.get(url.format(rounds_df['season'][index],rnd_num),timeout=None)
        json = r.json()
        try: 
            for item in json['MRData']['StandingsTable']['StandingsLists'][0]['ConstructorStandings']:
                try:
                    constructor_standings['season'].append(int(json['MRData']['StandingsTable']['StandingsLists'][0]['season']))
                except:
                    constructor_standings['season'].append(None)

                try:
                    constructor_standings['round'].append(int(json['MRData']['StandingsTable']['StandingsLists'][0]['round']))
                except:
                    constructor_standings['round'].append(None)

                try:
                    constructor_standings['position'].append(int(item['position']))
                except:
                    constructor_standings['position'].append(None)

                try:
                    constructor_standings['constructorId'].append(item['Constructor']['constructorId'])
                except:
                    constructor_standings['constructorId'].append(None)

                try:
                    constructor_standings['points'].append(float(item['points']))
                except:
                    constructor_standings['points'].append(None)

                try:
                    constructor_standings['wins'].append(int(item['wins']))
                except:
                    constructor_standings['wins'].append(None)
        except:
            pass

In [8]:
constructor_standings_df = pd.DataFrame(constructor_standings)
constructor_standings_df.to_csv('constructorStandings.csv')

In [9]:
constructor_standings_df.tail()

,season,round,position,constructorId,points,wins
13146,2023,22,6,alpine,120.0,0
13147,2023,22,7,williams,28.0,0
13148,2023,22,8,alphatauri,25.0,0
13149,2023,22,9,alfa,16.0,0
13150,2023,22,10,haas,12.0,0


# Constructor Info

In [19]:
constructor_info = {'season': [],
              'constructorId': [],
              'name': [],
              'nationality': []}

for year in list(range(1950,2024)):
    url = 'http://ergast.com/api/f1/{}/constructors.json'
    r = requests.get(url.format(year),timeout=None)
    json = r.json()
    try:
        for item in json['MRData']['ConstructorTable']['Constructors']:
            try:
                constructor_info['season'].append(int(json['MRData']['ConstructorTable']['season']))
            except:
                constructor_info['season'].append(None)
                
            try:
                constructor_info['constructorId'].append(item['constructorId'])
            except:
                constructor_info['constructorId'].append(None)
                
            try:
                constructor_info['name'].append(item['name'])
            except:
                constructor_info['name'].append(None)
                
            try:
                constructor_info['nationality'].append(item['nationality'])
            except:
                constructor_info['nationality'].append(None)
    except:
        print("Results Not Available")

In [20]:
constructor_info_df = pd.DataFrame(constructor_info)
constructor_info_df.to_csv('constructorInfo.csv')

# Driver Info 

In [21]:
constructorinfo = pd.read_csv('constructorInfo.csv')
driver_info = {'season': [],
              'driverId': [],
              'constructorId': [],
              'permanentNumber': [],
              'code': [],
              'givenName': [],
              'familyName': [],
              'dateOfBirth': [],
              'nationality': []}

for year in list(range(1950,2024)):
    for constructor in constructorinfo.loc[constructorinfo['season'] == year].constructorId.unique():
        url = 'http://ergast.com/api/f1/{}/constructors/{}/drivers.json'
        r = requests.get(url.format(year,constructor),timeout=None)
        json = r.json()
        try:
            for item in json['MRData']['DriverTable']['Drivers']:
                try:
                    driver_info['season'].append(int(json['MRData']['DriverTable']['season']))
                except:
                    driver_info['season'].append(None)

                try:
                    driver_info['driverId'].append(item['driverId'])
                except:
                    driver_info['driverId'].append(None)
                    
                try:
                    driver_info['constructorId'].append(json['MRData']['DriverTable']['constructorId'])
                except:
                    driver_info['constructorId'].append(None)

                try:
                    driver_info['permanentNumber'].append(int(item['permanentNumber']))
                except:
                    driver_info['permanentNumber'].append(None)

                try:
                    driver_info['code'].append(item['code'])
                except:
                    driver_info['code'].append(None)

                try:
                    driver_info['givenName'].append(item['givenName'])
                except:
                    driver_info['givenName'].append(None)

                try:
                    driver_info['familyName'].append(item['familyName'])
                except:
                    driver_info['familyName'].append(None)

                try:
                    driver_info['dateOfBirth'].append(item['dateOfBirth'])
                except:
                    driver_info['dateOfBirth'].append(None)

                try:
                    driver_info['nationality'].append(item['nationality'])
                except:
                    driver_info['nationality'].append(None)

        except:
            pass

In [22]:
driver_info_df = pd.DataFrame(driver_info)
driver_info_df.to_csv('driverInfo.csv')

In [23]:
driver_info_df.tail(22)

,season,driverId,constructorId,permanentNumber,code,givenName,familyName,dateOfBirth,nationality
3506,2023,bottas,alfa,77.0,BOT,Valtteri,Bottas,1989-08-28,Finnish
3507,2023,zhou,alfa,24.0,ZHO,Guanyu,Zhou,1999-05-30,Chinese
3508,2023,de_vries,alphatauri,21.0,DEV,Nyck,de Vries,1995-02-06,Dutch
3509,2023,lawson,alphatauri,40.0,LAW,Liam,Lawson,2002-02-11,New Zealander
3510,2023,ricciardo,alphatauri,3.0,RIC,Daniel,Ricciardo,1989-07-01,Australian
3511,2023,tsunoda,alphatauri,22.0,TSU,Yuki,Tsunoda,2000-05-11,Japanese
3512,2023,gasly,alpine,10.0,GAS,Pierre,Gasly,1996-02-07,French
3513,2023,ocon,alpine,31.0,OCO,Esteban,Ocon,1996-09-17,French
3514,2023,alonso,aston_martin,14.0,ALO,Fernando,Alonso,1981-07-29,Spanish
3515,2023,stroll,aston_martin,18.0,STR,Lance,Stroll,1998-10-29,Canadian
